In [ ]:
# ==============================================================================
# PROVOCATION BENCHMARK: LEVEL-BASED GRANGER CAUSALITY TEST & COMPARATIVE PLOTS
# EPU (Baker, Bloom & Davis) vs. VIX Tupiniquim (PCA & LASSO)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

print("--- [PROVOCATION BENCHMARK]: LEVEL-BASED ANALYSIS (VIX PCA & LASSO vs EPU) ---")

# ==============================================================================
# 1. HISTORICAL SERIES INGESTION AND ALIGNMENT
# ==============================================================================
# Load EPU series from Baker, Bloom & Davis database
df_epu = pd.read_excel('epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index')
df_epu['Data'] = pd.to_datetime(df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01')
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

# Load both VIX series (PCA & LASSO) generated in the main pipeline
df_vix = pd.read_excel('tabela_vix_tupiniquim_pca.xlsx') 
df_vix['Data'] = pd.to_datetime(df_vix['Data'])

# Structural Decomposition (STL) for EPU Filtering (Seasonal Adjustment)
print("Applying STL Structural Decomposition (period=13) to EPU series...")
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

# Merge and Sample Alignment
df_analise = pd.merge(df_vix, df_epu[['Data', 'EPU_SA']], on='Data', how='inner').sort_values('Data').reset_index(drop=True)
print(f"[ALIGNMENT]: Final common sample with {len(df_analise)} synchronized months.")

# Standardization of all series to Base Mean = 100 within the sample
df_analise['VIX_PCA_100'] = (df_analise['VIX_PCA'] / df_analise['VIX_PCA'].mean()) * 100
df_analise['VIX_LASSO_100'] = (df_analise['VIX_LASSO'] / df_analise['VIX_LASSO'].mean()) * 100
df_analise['EPU_Base100'] = (df_analise['EPU_SA'] / df_analise['EPU_SA'].mean()) * 100

# ==============================================================================
# 2. STATIONARITY VERIFICATION (ADF TEST IN LEVEL)
# ==============================================================================
print("\n--- AUGMENTED DICKEY-FULLER (ADF) TEST IN LEVEL ---")
p_vix_pca = adfuller(df_analise['VIX_PCA_100'])[1]
p_vix_lasso = adfuller(df_analise['VIX_LASSO_100'])[1]
p_epu = adfuller(df_analise['EPU_Base100'])[1]

print(f"-> P-value VIX Tupiniquim (PCA - Level):   {p_vix_pca:.4f} -> {'Stationary I(0)' if p_vix_pca < 0.05 else 'Marginally Stationary'}")
print(f"-> P-value VIX Tupiniquim (LASSO - Level): {p_vix_lasso:.4f} -> {'Stationary I(0)' if p_vix_lasso < 0.05 else 'Non-Stationary'}")
print(f"-> P-value EPU_SA (Level):                 {p_epu:.4f} -> {'Stationary I(0)' if p_epu < 0.05 else 'Non-Stationary'}")

# ==============================================================================
# 3. LEVEL-BASED GRANGER CAUSALITY TEST (Lags 1 to 3)
# ==============================================================================
max_lags = 3
print(f"\n--- RUNNING GRANGER CAUSALITY TEST IN LEVEL (Lags 1 to {max_lags}) ---")

def run_granger_test(data_df, var_y, var_x, h0_label):
    res = grangercausalitytests(data_df[[var_y, var_x]], maxlag=max_lags, verbose=False)
    rows = []
    for lag in range(1, max_lags + 1):
        f_stat = res[lag][0]['ssr_ftest'][0]
        p_val = res[lag][0]['ssr_ftest'][1]
        status = "Reject H0 (Granger-causes)" if p_val < 0.05 else "Fail to Reject H0"
        rows.append({
            'Causal Relationship (H0)': h0_label,
            'Lag': lag,
            'F-Statistic': f_stat,
            'p-value': p_val,
            'Conclusion (5% sig.)': status
        })
    return rows

table6_data = []

# 3.1 VIX (PCA) vs EPU
table6_data.extend(run_granger_test(df_analise, 'EPU_Base100', 'VIX_PCA_100', 'VIX (PCA) ↛ EPU (VIX PCA does not Granger-cause News)'))
table6_data.extend(run_granger_test(df_analise, 'VIX_PCA_100', 'EPU_Base100', 'EPU ↛ VIX (PCA) (News does not Granger-cause VIX PCA)'))

# 3.2 VIX (LASSO) vs EPU
table6_data.extend(run_granger_test(df_analise, 'EPU_Base100', 'VIX_LASSO_100', 'VIX (LASSO) ↛ EPU (VIX LASSO does not Granger-cause News)'))
table6_data.extend(run_granger_test(df_analise, 'VIX_LASSO_100', 'EPU_Base100', 'EPU ↛ VIX (LASSO) (News does not Granger-cause VIX LASSO)'))

df_table6_en = pd.DataFrame(table6_data)

print("\n" + "="*95)
print("     TABLE 6: LEVEL-BASED GRANGER CAUSALITY TEST (VIX I vs. BRAZIL EPU)")
print("="*95)
print(df_table6_en.to_string(index=False, formatters={
    'F-Statistic': '{:.4f}'.format,
    'p-value': '{:.4f}'.format
}))
print("="*95)
df_table6_en.to_excel('table_6_vix1_granger_level_en.xlsx', index=False)

# ==============================================================================
# 4. ISOLATED DUAL-AXIS PLOTS (VIA NEGATIVA: NO LEGEND)
# ==============================================================================

# --- 4.1. PLOT 1: VIX (PCA) vs EPU (DUAL-AXIS) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

color1 = 'navy'
ax1.set_xlabel('Years', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via PCA (Base Mean = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_PCA_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Brazil EPU Index (Base Mean = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_pca_vs_epu_en.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n[IMAGE SAVED]: 'vix_pca_vs_epu_en.png' generated successfully!")


# --- 4.2. PLOT 2: VIX (LASSO) vs EPU (DUAL-AXIS) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

color1 = 'firebrick'
ax1.set_xlabel('Years', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via LASSO (Base Mean = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_LASSO_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Brazil EPU Index (Base Mean = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_lasso_vs_epu_en.png', dpi=300, bbox_inches='tight')
plt.show()
print("[IMAGE SAVED]: 'vix_lasso_vs_epu_en.png' generated successfully!")

print("\n--- ANALYSIS COMPLETED SUCCESSFULLY! ---")